<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03a_visualizations_censustract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sub Question 1: Which neighborhoods (and census tracts) experienced the highest rates of business closures during 2020-2021, and which show the most new openings (recovery) from 2022 to 2025?**

## Set Up

In [1]:
!pip install contextily

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

#added more that we use in lab
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls /content/drive
!ls /content/drive/Shareddrives

MyDrive  Shareddrives


In [43]:
import pandas as pd

gdf = pd.read_csv("/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cleaned/rbl_total_after_2016_cleaned.csv")

In [6]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148940 entries, 0 to 148939
Data columns (total 40 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         148940 non-null  int64  
 1   uniqueid                           148940 non-null  object 
 2   business_account_number            148940 non-null  int64  
 3   location_id                        148940 non-null  object 
 4   ownership_name                     148940 non-null  object 
 5   dba_name                           148787 non-null  object 
 6   street_address                     148940 non-null  object 
 7   city                               148940 non-null  object 
 8   state                              148934 non-null  object 
 9   source_zipcode                     148931 non-null  float64
 10  business_start_date                148940 non-null  object 
 11  business_end_date                  7551

In [7]:
# # adding another plot and importing a shpfile of SF so we can see where the points are in SF

# import matplotlib.pyplot as plt

# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# eproject to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

## Plotting Closings vs. Openings over time for all businesses

In [8]:
import plotly.express as px

# counting number of opening and closings - Abigail
counts = gdf.groupby(["year", "status"]).size().reset_index(name="count")
counts = counts[counts["year"] <= 2025]

# Making graph of this
fig = px.line(
    counts,
    x="year",
    y="count",
    color="status",
    markers=True
)

# Adding labels
fig.update_layout(
    title="San Francisco Business Openings vs Closings (2016–2025)",
    xaxis_title="Year",
    yaxis_title="Number of Businesses",
    legend_title="Status"
)

#adding code to show each year on x-axis
fig.update_xaxes(dtick=1)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

#making background white and removing vert lines
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray"
)

fig.show()


## **Tract Level Analysis**

In [9]:
#I added a shp file with census tracts to our folder

tracts_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cb_2020_06_tract_500k"
)

In [10]:
#checking to make sure all good

tracts_gdf.columns

# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME","GEOID", "geometry"]]


In [11]:
# was getting error with "index_right" - removing from both dfs
tracts_gdf = tracts_gdf.drop(columns=["index_right", ], errors="ignore")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

In [12]:
# Joining the gdf and tract gdf so we can summarize within tracts

gdf = gdf.set_geometry(
    gpd.points_from_xy(gdf.lon, gdf.lat)
)

gdf = gdf.set_crs(epsg=4326)
tracts_gdf = tracts_gdf.to_crs(epsg=4326)

# joining within
business_tracts_gdf = gpd.sjoin(gdf, tracts_gdf, how="left", predicate="within")

In [13]:
business_tracts_gdf = business_tracts_gdf[(business_tracts_gdf["year"] >= 2016) & (business_tracts_gdf["year"] <= 2025)]

In [14]:
business_tracts_gdf.columns.tolist()

cols_to_keep = [
    "uniqueid",
    "naics_code",
    "year",
    "status",
    "location_start_date",
    "location_end_date",
    "GEOID",
    "geometry",
    "lon", "lat"
]

business_tracts_gdf = business_tracts_gdf[cols_to_keep]

In [15]:
# I'm counting the number of businesses per tract per year, separated by closing and openings status
tract_year = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
    .pivot(index=["GEOID", "year"], columns="status", values="count")
    .fillna(0)
    .reset_index()
)

In [16]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    tract_year,
    on="GEOID",
    how="left"
).fillna(0)

In [17]:
tracts_plot.info()
tracts_plot = gpd.clip(tracts_plot, sf_poly)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 11301 entries, 0 to 11300
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   GEOID     11301 non-null  object  
 1   geometry  11301 non-null  geometry
 2   year      11301 non-null  float64 
 3   closed    11301 non-null  float64 
 4   opened    11301 non-null  float64 
dtypes: float64(3), geometry(1), object(1)
memory usage: 441.6+ KB


In [18]:
tracts_plot["year"] = tracts_plot["year"].astype(int)
tracts_plot = tracts_plot.sort_values(["GEOID", "year"])

In [19]:
print(tracts_plot.columns)

Index(['GEOID', 'geometry', 'year', 'closed', 'opened'], dtype='object')


In [20]:
# opened = tracts_plot[tracts_plot["status"] == "opened"]
# closed = tracts_plot[tracts_plot["status"] == "closed"]

In [21]:
# This is the code I used to make some interactive visualizations, but the file size was preventing me from uploading to Github. I commented this out for now so I could upload.

# fig_closed = px.choropleth_mapbox(
#     closed,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Reds",
#     height=600
# )

# fig_closed.update_layout(title="Business Closings by Census Tract (2016–2025)")
# fig_closed.show()

In [22]:
# fig_opened = px.choropleth_mapbox(
#     opened,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Blues",
#     height=600
# )

# fig_opened.update_layout(title="Business Openings by Census Tract (2016–2025)")
# fig_opened.show()

Census Tract Analysis - Baselining

In [23]:
# Reformatting so each row is a unique count per tract, year, and status (so we don't have closings and openings for business in one row)
business_tracts_separated = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
)

In [24]:
# Baselining by 2016

baseline = business_tracts_separated[business_tracts_separated["year"] == 2016][["GEOID", "status", "count"]].rename(columns={"count": "baseline"}) # makes new df and filtering for only 2016 so we can use as baseline
business_tracts_separated = business_tracts_separated.merge(baseline, on=["GEOID", "status"], how="left") # joining this new baseline df to business tracts df so we can calculate change
business_tracts_separated["change_from_2016"] = business_tracts_separated["count"] - business_tracts_separated["baseline"] # subtracts change from baseline


In [25]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    business_tracts_separated,
    on="GEOID",
    how="left"
).fillna(0)

In [26]:
opened = tracts_plot[tracts_plot["status"] == "opened"]
closed = tracts_plot[tracts_plot["status"] == "closed"]

In [27]:
# sanity check on changes

opened["change_from_2016"].describe()

# most tracts didn't have major changes (average of 3 fewer openings compared to 2016)
# but some tracts had a major change (both pos and neg compared to 2016)- these extremes will skew our viz

,change_from_2016
count,2435.000000
mean,-3.345380
std,29.844506
min,-504.000000
25%,-10.000000
50%,0.000000
75%,4.000000
max,426.000000


In [28]:
closed["change_from_2016"].describe()

,change_from_2016
count,2416.000000
mean,14.512831
std,25.969480
min,-11.000000
25%,3.000000
50%,10.000000
75%,18.000000
max,510.000000


In [29]:
# fig = px.choropleth_mapbox(
#     opened,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="Blues",
#     range_color=[-20, 20], # adding this so that the legend scale doesn't change each year - need to make decision around this bc min and max will not show most changes
#     height=600,
# )

# fig.update_layout(title="SF Business Openings Change from 2016 by Census Tract")
# fig.show()

In [30]:
fig.write_html("sf_business_map.html")

In [31]:
from google.colab import files
files.download("sf_business_map.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
# fig = px.choropleth_mapbox(
#     closed,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="Reds",
#     range_color=[-10,20],
#     height=600,
# )

# fig.update_layout(title="SF Business Closings Change from 2016 by Census Tract")
# fig.show()

## Neighborhood Level Analysis

In [33]:
# Adding SF neighborhood geometry

nbhd_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/SF_neighborhoods_geom"
)

In [37]:
nbhd_gdf.drop(columns=["link"], inplace=True)

In [39]:
nbhd_gdf.columns

Index(['name', 'geometry'], dtype='object')

In [44]:
# Joining the gdf and nbhd gdf so we can summarize within nbhds

gdf = gdf.set_geometry(
    gpd.points_from_xy(gdf.lon, gdf.lat)
)

gdf = gdf.set_crs(epsg=4326)
nbhd_gdf = nbhd_gdf.to_crs(epsg=4326)

# joining within
business_neighborhood_gdf = gpd.sjoin(gdf, nbhd_gdf, how="left", predicate="within")

In [47]:
business_neighborhood_gdf.columns

Index(['Unnamed: 0', 'uniqueid', 'business_account_number', 'location_id',
       'ownership_name', 'dba_name', 'street_address', 'city', 'state',
       'source_zipcode', 'business_start_date', 'business_end_date',
       'location_start_date', 'location_end_date', 'administratively_closed',
       'mail_address', 'mail_city', 'mail_state', 'mail_zipcode', 'naics_code',
       'naics_code_description', 'naics_code_descriptions_list', 'lic_code',
       'lic_code_description', 'lic_code_descriptions_list', 'parking_tax',
       'transient_occupancy_tax', 'business_location', 'business_corridor',
       'neighborhoods_analysis_boundaries', 'supervisor_district',
       'community_benefit_district', 'data_as_of', 'data_loaded_at',
       'geometry', 'administratively_closed_bool', 'year', 'status', 'lon',
       'lat', 'index_right', 'name'],
      dtype='object')